# 01 · Entity identification model

The complete **Part 1: entity identification** of our architecture, plus training.
Use **Run All**. If a trained model (`entity_identifier.pt`) already exists, it is
**loaded instead of retrained**, unless `RETRAIN = True` in the settings.

| Notebook | Contents |
|---|---|
| **01_entity_model** (this one) | settings, helpers, dataset loaders, the model, training |
| 02_tests | unit tests, data checks, smoke test |
| 03_evaluation | training curve, threshold tuning, test scores, reworded labels, demo |

Cells marked *definitions* (sections 1–6) are also loaded by notebooks 02 and 03,
so the model code is written only once, here.

Expected folder layout (`data/raw` may also sit one or two levels up):
```
entity_extraction/
├── 01_entity_model.ipynb
├── 02_tests.ipynb
├── 03_evaluation.ipynb
├── entity_identifier.pt          <- created by training (or downloaded)
└── data/raw/REDocRed/...         <- from the team's data.zip
```

## 1. Setup *(definitions)*

Needs `torch` and `transformers`. If this fails with *No module named ...*, uncomment
the `%pip` line, run it once and restart the kernel. For GPU training,
`GPU available` must be `True`; otherwise install the CUDA build of PyTorch from
https://pytorch.org/get-started/locally/.

In [ ]:
# %pip install torch transformers matplotlib

import json, math, os, random, re, statistics, time
from dataclasses import dataclass, field
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F
import transformers
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch", torch.__version__, "| transformers", transformers.__version__)
print("GPU available:", torch.cuda.is_available(),
      f"({torch.cuda.get_device_name(0)})" if torch.cuda.is_available() else "")

## 2. Settings *(definitions)*

The cell to edit. Typical changes:
- `RETRAIN = True` to train again although a saved model exists
- `QUICK = True` for a 1-epoch trial on 40 documents
- `DATASETS = ["redocred", "scierc"]` to add datasets
- another backbone: `MODEL_NAME = "microsoft/deberta-v3-base"` **and** a new
  `CHECKPOINT` name, e.g. `"entity_identifier_deberta.pt"` (DeBERTa also needs
  `%pip install sentencepiece protobuf`)
- `THRESHOLD`: set it to the best value found in notebook 03

In [ ]:
CONFIG = dict(
    DATASETS=["redocred"],            # any of: "redocred", "scierc", "dwie"
    MODEL_NAME="bert-base-cased",     # encoder backbone (must match the checkpoint)
    CHECKPOINT="entity_identifier.pt",
    RETRAIN=False,                    # False: load CHECKPOINT if it exists, else train
    QUICK=False,                      # True: 40 training docs, 1 epoch

    # training
    EPOCHS=10, BATCH_SIZE=4, ENCODER_LR=2e-5, HEAD_LR=1e-4,
    DROPOUT=0.1, POS_WEIGHT=None, SEED=42, LOG_EVERY=50,
    MAX_SPAN_WIDTH=8, RESTRICT_SPANS_TO_SENTENCES=True, DWIE_DEV_SIZE=100,

    # prediction
    THRESHOLD=0.5,                    # keep a span if its best label scores above this
)

HERE = Path.cwd()


def find_data_dir():
    for base in [HERE, HERE.parent, HERE.parent.parent]:
        if (base / "data" / "raw").is_dir():
            return base / "data" / "raw"
    return None


DATA_DIR = find_data_dir()
CHECKPOINT_PATH = HERE / CONFIG["CHECKPOINT"]
LOG_PATH = CHECKPOINT_PATH.with_name(CHECKPOINT_PATH.stem + "_log.json")
print("data folder:", DATA_DIR)
print("checkpoint :", CHECKPOINT_PATH.name,
      "(exists)" if CHECKPOINT_PATH.exists() else "(not trained yet)")

## 3. Helpers *(definitions)*

Tokenising text with character positions, listing candidate spans (1–8 tokens),
building 0/1 training targets, choosing which spans to keep, and strict entity F1.

In [ ]:
"""
Part 1 (entity identification): pure-Python helpers.

No PyTorch needed here, so these can be unit-tested on their own.
  - tokenize_with_offsets: raw text -> word tokens with character offsets
  - enumerate_spans:       every run of 1..max_width consecutive tokens
  - char_to_word_span:     convert character-offset annotations (e.g. DWIE)
  - build_entity_targets:  gold mentions -> 0/1 target matrix [spans x labels]
  - decode_mentions:       probabilities -> kept mentions (the "Keep entity spans" box)

Conventions used throughout:
  - Token (word) spans are (start, end) with `end` INCLUSIVE, as in the diagram
    ('Northstar Labs' = tokens 2-3).
  - Character spans are (start, end) with `end` EXCLUSIVE, Python-slice style
    (text[13:27] == 'Northstar Labs').
"""

import re
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Sequence, Tuple

Span = Tuple[int, int]

# In order: dotted abbreviations (U.S.), words / numbers (well-known, 1,000, 3.5),
# clitics split off their word (Germany's -> Germany + 's), single punctuation.
_TOKEN_RE = re.compile(
    r"(?:[A-Za-z]\.){2,}|\w+(?:[-.,&]\w+)*|['’]\w+|[^\w\s]", re.UNICODE)
_SENT_END = {".", "!", "?"}
# Tokens after which a '.' does not end the sentence.
_ABBREVIATIONS = {"mr", "mrs", "ms", "dr", "prof", "st", "jr", "sr", "gen", "lt",
                  "col", "sgt", "capt", "gov", "sen", "rep", "pres", "no", "vs",
                  "etc", "inc", "ltd", "co", "corp", "jan", "feb", "mar", "apr",
                  "jun", "jul", "aug", "sep", "sept", "oct", "nov", "dec", "e.g", "i.e"}


@dataclass
class Document:
    """One document, either from raw text or from a pre-tokenised dataset.

    words         : word tokens
    char_offsets  : (start, end) character span of each word in `text`
                    (None for pre-tokenised datasets without raw text)
    sentence_ids  : sentence index of each word
    gold_mentions : training only; dicts {"start", "end", "label"} over word
                    indices, `end` inclusive
    entity_labels : the label set this document is scored against
                    (training: its own dataset's labels; run time: user's labels)
    multi_label   : if True, a span may carry several correct labels at once
                    (DWIE: 'Germany' is a location, a gpe and a gpe0), so
                    decoding keeps every label above the threshold
    """
    words: List[str]
    entity_labels: List[str]
    text: Optional[str] = None
    char_offsets: Optional[List[Span]] = None
    sentence_ids: Optional[List[int]] = None
    gold_mentions: List[Dict] = field(default_factory=list)
    doc_id: str = ""
    dataset: str = ""
    multi_label: bool = False


def tokenize_with_offsets(text: str, split_points: Optional[Sequence[int]] = None
                          ) -> Tuple[List[str], List[Span], List[int]]:
    """Split raw text into word tokens, remembering where each sits in the text.

    Returns (words, char_offsets, sentence_ids).

    `split_points` (optional): character positions where a token must be cut,
    e.g. the begin/end of every annotated mention, so each mention lines up
    exactly with whole tokens ('ZDF-Politbarometer' annotated as 'ZDF').

    A sentence ends after '.', '!' or '?' when the next token starts with a
    capital letter, digit or quote, unless the '.' follows an abbreviation
    such as 'Dr' or 'U.S.'.
    """
    offsets = [(m.start(), m.end()) for m in _TOKEN_RE.finditer(text)]
    if split_points:
        cuts = sorted(set(split_points))
        refined = []
        for s, e in offsets:
            inner = [c for c in cuts if s < c < e]
            bounds = [s] + inner + [e]
            refined.extend((a, b) for a, b in zip(bounds, bounds[1:]) if text[a:b].strip())
        offsets = refined
    words = [text[s:e] for s, e in offsets]

    sent_ids, sent = [], 0
    for i, w in enumerate(words):
        sent_ids.append(sent)
        if w in _SENT_END and i + 1 < len(words):
            prev = words[i - 1].lower() if i > 0 else ""
            nxt = words[i + 1]
            # '.' directly attached to an abbreviation (no space between)
            attached = i > 0 and offsets[i - 1][1] == offsets[i][0]
            # abbreviation ('Dr.') or a single-letter initial ('J. Smith')
            is_abbrev = w == "." and attached and (
                prev in _ABBREVIATIONS or (len(prev) == 1 and prev.isalpha()))
            if not is_abbrev and (nxt[0].isupper() or nxt[0].isdigit() or nxt[0] in "\"'“‘"):
                sent += 1
    return words, offsets, sent_ids


def document_from_text(text: str, entity_labels: Sequence[str], doc_id: str = "") -> Document:
    """Build a Document from raw text (run-time / demo input)."""
    words, offsets, sent_ids = tokenize_with_offsets(text)
    return Document(words=words, entity_labels=list(entity_labels), text=text,
                    char_offsets=offsets, sentence_ids=sent_ids, doc_id=doc_id)


def enumerate_spans(n_words: int, max_width: int = 8,
                    sentence_ids: Optional[Sequence[int]] = None) -> List[Span]:
    """Every run of 1..max_width consecutive tokens.

    If `sentence_ids` is given, spans that cross a sentence boundary are skipped
    (no entity crosses one, and this removes many useless candidates).
    The diagram's count of 52 spans for a 10-token text is without this filter.
    """
    spans = []
    for start in range(n_words):
        for end in range(start, min(start + max_width, n_words)):
            if sentence_ids is not None and sentence_ids[end] != sentence_ids[start]:
                break
            spans.append((start, end))
    return spans


def char_to_word_span(char_start: int, char_end: int,
                      char_offsets: Sequence[Span]) -> Optional[Span]:
    """Map a character span (end exclusive) to an inclusive word span.

    Returns None if no word overlaps the character span.
    """
    covered = [i for i, (s, e) in enumerate(char_offsets) if s < char_end and e > char_start]
    if not covered:
        return None
    return covered[0], covered[-1]


def build_entity_targets(spans: Sequence[Span], gold_mentions: Sequence[Dict],
                         entity_labels: Sequence[str]) -> Tuple[List[List[float]], Dict[str, int]]:
    """0/1 target matrix of shape [len(spans), len(entity_labels)].

    A span gets 1 for each label it is annotated with; every other span and
    label is 0 ("spans with no annotated label: every label = 0").

    Also returns counts of gold mentions that cannot be learned, so you can
    report them: those wider than max_width (no candidate span) and those whose
    label is not in this document's label set.
    """
    span_index = {s: i for i, s in enumerate(spans)}
    label_index = {l: j for j, l in enumerate(entity_labels)}
    targets = [[0.0] * len(entity_labels) for _ in spans]
    stats = {"gold": 0, "matched": 0, "too_long_or_filtered": 0, "unknown_label": 0}

    for m in gold_mentions:
        stats["gold"] += 1
        j = label_index.get(m["label"])
        if j is None:
            stats["unknown_label"] += 1
            continue
        i = span_index.get((m["start"], m["end"]))
        if i is None:
            stats["too_long_or_filtered"] += 1
            continue
        targets[i][j] = 1.0
        stats["matched"] += 1
    return targets, stats


def entity_prf(predicted: Sequence[Sequence[Dict]], gold: Sequence[Sequence[Dict]]) -> Dict[str, float]:
    """Strict entity precision / recall / F1, micro-averaged over documents.

    A predicted mention is correct only if its start, end AND label all match
    a gold mention of the same document. Inputs are per-document lists of
    mention dicts with "start", "end", "label". Returns percentages.
    """
    tp = n_pred = n_gold = 0
    for pred_doc, gold_doc in zip(predicted, gold):
        p = {(m["start"], m["end"], m["label"]) for m in pred_doc}
        g = {(m["start"], m["end"], m["label"]) for m in gold_doc}
        tp += len(p & g)
        n_pred += len(p)
        n_gold += len(g)
    precision = 100.0 * tp / n_pred if n_pred else 0.0
    recall = 100.0 * tp / n_gold if n_gold else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {"precision": precision, "recall": recall, "f1": f1}


def decode_mentions(spans: Sequence[Span], probs: Sequence[Sequence[float]],
                    entity_labels: Sequence[str], doc: Document,
                    threshold: float = 0.5) -> List[Dict]:
    """Keep every span with at least one label above `threshold`, labelled with
    its highest-scoring label. Nested / overlapping spans can both be kept.

    If `doc.multi_label` is True, a span instead yields one mention for EVERY
    label above the threshold (all of them share the same span_idx).

    Returns mention dicts sorted by position:
      {"span_idx", "start", "end", "text", "label", "score", "char_start", "char_end"}
    `span_idx` indexes the candidate span list, so callers can pick out the
    matching span vector for coreference.
    """
    mentions = []
    for i, (span, row) in enumerate(zip(spans, probs)):
        if doc.multi_label:
            chosen = [j for j, p in enumerate(row) if p > threshold]
        else:
            best_j = max(range(len(row)), key=row.__getitem__)
            chosen = [best_j] if row[best_j] > threshold else []
        start, end = span
        for j in chosen:
            m = {
                "span_idx": i,
                "start": start,
                "end": end,
                "label": entity_labels[j],
                "score": float(row[j]),
            }
            if doc.char_offsets is not None and doc.text is not None:
                cs, ce = doc.char_offsets[start][0], doc.char_offsets[end][1]
                m.update(char_start=cs, char_end=ce, text=doc.text[cs:ce])
            else:
                m["text"] = " ".join(doc.words[start:end + 1])
            mentions.append(m)
    mentions.sort(key=lambda m: (m["start"], m["end"], -m["score"]))
    return mentions

## 4. Dataset loaders *(definitions)*

Turn SciERC, Re-DocRED and DWIE into one `Document` format. Each document is scored
only against **its own dataset's labels**, and label names are made readable
(`PER` → *person*) because the label encoder reads them as words. `load_split`
returns the train, dev or test documents of the datasets chosen in the settings.

In [ ]:
"""
Loaders that turn SciERC, DWIE and Re-DocRED files into ee_utils.Document
objects for entity identification.

Each document gets its OWN dataset's entity label set ("each document is
scored only against its own dataset's label set"). Label names are converted
to readable text (e.g. DocRED 'PER' -> 'person'), because the label encoder
reads the label as words: 'person' means something to BERT, 'PER' much less.

Expected file formats (check these against your downloaded copies):
  SciERC     processed JSON lines (as in SciIE / DyGIE++):
             {"doc_key", "sentences": [[tok, ...], ...],
              "ner": [[[start, end, label], ...] per sentence], ...}
             token indices are document-level, end INCLUSIVE.
  Re-DocRED  one JSON list of documents:
             {"title", "sents": [[tok, ...], ...],
              "vertexSet": [[{"name", "sent_id", "pos": [start, end], "type"}, ...], ...]}
             pos is sentence-level, end EXCLUSIVE.
  DWIE       one JSON file per article:
             {"id", "content", "mentions": [{"begin", "end", "concept"}, ...],
              "concepts": [{"concept", "tags": ["type::person", ...]}, ...]}
             begin/end are character offsets, end EXCLUSIVE.
"""

import json
import os
from typing import Dict, Iterable, List


# Readable label names (edit freely; this is what the label encoder reads).
SCIERC_LABELS = {
    "Task": "task", "Method": "method", "Metric": "metric", "Material": "material",
    "OtherScientificTerm": "other scientific term", "Generic": "generic",
}
DOCRED_LABELS = {
    "PER": "person", "ORG": "organization", "LOC": "location",
    "TIME": "time", "NUM": "number", "MISC": "miscellaneous",
}


def _readable(tag: str) -> str:
    """'type::person' -> 'person', 'sport_team' -> 'sport team'."""
    return tag.split("::", 1)[-1].replace("_", " ").strip()


# ---------------------------------------------------------------- SciERC ----

def load_scierc(path: str) -> List[Document]:
    docs = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            ex = json.loads(line)
            words, sent_ids = [], []
            for s, sent in enumerate(ex["sentences"]):
                words.extend(sent)
                sent_ids.extend([s] * len(sent))
            mentions = [{"start": st, "end": en, "label": SCIERC_LABELS.get(lab, _readable(lab))}
                        for sent_ner in ex.get("ner", []) for st, en, lab in sent_ner]
            docs.append(Document(words=words, sentence_ids=sent_ids, gold_mentions=mentions,
                                 entity_labels=sorted(set(SCIERC_LABELS.values())),
                                 doc_id=ex.get("doc_key", ""), dataset="scierc"))
    return docs


# ------------------------------------------------------------- Re-DocRED ----

def load_redocred(path: str) -> List[Document]:
    with open(path, encoding="utf-8") as f:
        data = json.load(f)
    docs = []
    for ex in data:
        words, sent_ids, sent_start = [], [], []
        for s, sent in enumerate(ex["sents"]):
            sent_start.append(len(words))
            words.extend(sent)
            sent_ids.extend([s] * len(sent))
        mentions, seen = [], set()
        for entity in ex.get("vertexSet", []):
            for m in entity:
                start = sent_start[m["sent_id"]] + m["pos"][0]
                end = sent_start[m["sent_id"]] + m["pos"][1] - 1      # -> inclusive
                label = DOCRED_LABELS.get(m["type"], _readable(m["type"]))
                if (start, end, label) not in seen:
                    seen.add((start, end, label))
                    mentions.append({"start": start, "end": end, "label": label})
        docs.append(Document(words=words, sentence_ids=sent_ids, gold_mentions=mentions,
                             entity_labels=sorted(set(DOCRED_LABELS.values())),
                             doc_id=ex.get("title", ""), dataset="redocred"))
    return docs


# ------------------------------------------------------------------ DWIE ----

# 'type::entity' is on almost every concept and says nothing, so it is dropped.
# The remaining 174 DWIE type tags are the label set (matching the proposal).
DWIE_SKIP_TYPES = {"entity"}


def _dwie_entity_types(concept: Dict) -> List[str]:
    """All readable 'type::' tags of a DWIE concept.

    DWIE types are hierarchical and multi-label, e.g. Germany is tagged
    entity, gpe, gpe0 and location. Every tag is a correct label, so all are
    kept: keeping only one would train the model that Germany is NOT a gpe.
    """
    types = [_readable(t) for t in (concept.get("tags") or []) if t.startswith("type::")]
    return [t for t in types if t not in DWIE_SKIP_TYPES]


def load_dwie(directory: str, split: str = None, label_set: List[str] = None) -> List[Document]:
    """Load every DWIE article JSON in `directory`.

    Use the `annos_with_content` folder made by DWIE's download script; the
    plain `annos` folder has no article text.

    If `split` is given ('train' or 'test'), keep only articles tagged with it
    (702 train, 100 test). DWIE has no validation split: hold out documents
    from 'train' yourself.

    `label_set` defaults to every type tag found in the loaded articles. When
    loading 'test', pass the training label set so both use the same labels.
    """
    raw = []
    for name in sorted(os.listdir(directory)):
        if name.endswith(".json"):
            with open(os.path.join(directory, name), encoding="utf-8") as f:
                ex = json.load(f)
            if split is None or split in ex.get("tags", []):
                raw.append(ex)
    if raw and raw[0].get("content") is None:
        raise ValueError(f"{directory} has no article text. Run DWIE's "
                         "src/dwie_download.py and use data/annos_with_content.")

    if label_set is None:
        label_set = sorted({t for ex in raw for c in ex.get("concepts", [])
                            for t in _dwie_entity_types(c)})
    docs = []
    for ex in raw:
        text = ex["content"]
        # Cut tokens at every mention boundary so each mention maps to whole words.
        cuts = [p for m in ex.get("mentions", []) for p in (m["begin"], m["end"])]
        words, offsets, sent_ids = tokenize_with_offsets(text, split_points=cuts)
        concept_types = {c["concept"]: _dwie_entity_types(c) for c in ex.get("concepts", [])}
        mentions, seen = [], set()
        for m in ex.get("mentions", []):
            span = char_to_word_span(m["begin"], m["end"], offsets)   # end is exclusive
            if span is None:
                continue
            for label in concept_types.get(m["concept"], []):
                if (span, label) not in seen:
                    seen.add((span, label))
                    mentions.append({"start": span[0], "end": span[1], "label": label})
        docs.append(Document(words=words, text=text, char_offsets=offsets,
                             sentence_ids=sent_ids, gold_mentions=mentions,
                             entity_labels=list(label_set), doc_id=ex.get("id", ""),
                             dataset="dwie", multi_label=True))
    return docs


def batches(docs: List[Document], batch_size: int) -> Iterable[List[Document]]:
    """Consecutive batches; shuffle `docs` beforehand to mix datasets."""
    for i in range(0, len(docs), batch_size):
        yield docs[i:i + batch_size]


def load_split(split):
    """split: 'train', 'dev' or 'test'. Returns {dataset name: [Document, ...]}."""
    if DATA_DIR is None:
        raise FileNotFoundError("No data/raw folder found next to this notebook "
                                "(or one/two levels up). Unzip the team's data.zip there.")
    files = {"scierc": {"train": "sciREC/train.json", "dev": "sciREC/dev.json",
                        "test": "sciREC/test.json"},
             "redocred": {"train": "REDocRed/train_revised.json",
                          "dev": "REDocRed/dev_revised.json",
                          "test": "REDocRed/test_revised.json"}}
    loaders = {"scierc": load_scierc, "redocred": load_redocred}
    sets = {}
    for name in CONFIG["DATASETS"]:
        if name in files:
            sets[name] = loaders[name](str(DATA_DIR / files[name][split]))
        elif name == "dwie":   # no official dev split: hold out the last training articles
            dwie_dir = str(DATA_DIR / "DWIE/data/annos_with_content")
            train = load_dwie(dwie_dir, split="train")
            n = CONFIG["DWIE_DEV_SIZE"]
            if split == "train":
                sets["dwie"] = train[:-n]
            elif split == "dev":
                sets["dwie"] = train[-n:]
            else:
                sets["dwie"] = load_dwie(dwie_dir, split="test", label_set=train[0].entity_labels)
    return sets

## 5. The model *(definitions)*

- `WordEncoder`: BERT → one vector per word (first subword); documents longer than
  512 tokens are encoded in overlapping windows.
- `SpanRepresentation`: span vector = [start ; end ; attention-pooled] = 3 × 768 = 2304.
- `EntityIdentifier`: span projection 2304 → 150 → ReLU → 768, label encoder (shared
  BERT) + projection 768 → 768, dot product → sigmoid per label, binary
  cross-entropy loss; `predict` keeps spans above the threshold.

In [ ]:
"""
Part 1 (entity identification): the PyTorch model.

Implements the "PART 1: Entity identification" block of the architecture,
up to and including "Keep entity spans" (coreference is a separate module):

  Raw text -> word tokens with positions                      (ee_utils)
  -> pretrained BERT encoder -> contextual vector per word     WordEncoder
  -> candidate span enumeration (1..8 tokens)                  (ee_utils)
  -> span vector [start ; end ; attention-pooled] = 3 x 768    SpanRepresentation
  -> span projection 2304 -> 150 -> ReLU -> 768                EntityIdentifier.span_proj
  -> label encoder (shared BERT) + linear 768 -> 768           EntityIdentifier.label_proj
  -> dot product -> sigmoid per label                          EntityIdentifier.forward
  -> keep spans with any label > 0.5, highest label            (ee_utils.decode_mentions)
  -> loss: binary cross-entropy per entity label

Outputs for teammates:
  - out["span_vectors"]   : vector of every candidate span (coreference, relations)
  - out["mention_vectors"]: vectors of the kept mentions (predict only)
  - model.represent_spans : vectors for any spans, e.g. gold mentions during training
  - model.encode_label_text: shared BERT label encoder, reusable for relation labels
"""

from typing import Dict, List, Optional, Sequence

import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModel, AutoTokenizer



class WordEncoder(nn.Module):
    """Pretrained BERT -> one contextual vector per word.

    Each word's vector is the vector of its FIRST subword ('biosensors' ->
    'bio', '##sens', '##ors' uses 'bio'). Documents longer than the 512-token
    limit are split into overlapping windows; each subword takes its vector
    from the window in which it sits furthest from an edge.
    """

    def __init__(self, model_name: str = "bert-base-cased", max_length: int = 512,
                 window_overlap: int = 128):
        super().__init__()
        # RoBERTa-style tokenizers need add_prefix_space for pre-split words.
        # DeBERTa-v3 needs `pip install sentencepiece protobuf` the first time.
        extra = {"add_prefix_space": True} if "roberta" in model_name.lower() else {}
        self.tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True, **extra)
        self.bert = AutoModel.from_pretrained(model_name)
        self.hidden_size = self.bert.config.hidden_size
        # Every window is [CLS] + subwords + [SEP]. The ids are read directly
        # because newer transformers versions removed the helper methods.
        # (For RoBERTa-style models, cls/sep are <s> and </s>.)
        self.cls_id = self.tokenizer.cls_token_id
        self.sep_id = self.tokenizer.sep_token_id
        if self.cls_id is None or self.sep_id is None:
            raise ValueError(f"{model_name}: tokenizer has no [CLS]/[SEP] tokens")
        self.window_size = max_length - 2
        self.window_overlap = window_overlap

    @property
    def device(self) -> torch.device:
        return next(self.bert.parameters()).device

    def forward(self, words: Sequence[str]) -> torch.Tensor:
        """Returns [n_words, hidden_size]."""
        if len(words) == 0:
            return torch.zeros(0, self.hidden_size, device=self.device)
        enc = self.tokenizer(list(words), is_split_into_words=True, add_special_tokens=False)
        subword_ids = enc["input_ids"]
        word_ids = enc.word_ids()

        # Index of each word's first subword. A word that produces no subwords
        # (rare, e.g. a stray control character) borrows the previous subword.
        first_subword = [-1] * len(words)
        for pos, w in enumerate(word_ids):
            if w is not None and first_subword[w] == -1:
                first_subword[w] = pos
        last = 0
        for i, pos in enumerate(first_subword):
            if pos == -1:
                first_subword[i] = last
            else:
                last = pos

        if len(subword_ids) == 0:  # empty document
            return torch.zeros(len(words), self.hidden_size, device=self.device)

        subword_vectors = self._encode_subwords(subword_ids)            # [n_sub, d]
        return subword_vectors[torch.tensor(first_subword, device=self.device)]

    def _encode_subwords(self, subword_ids: List[int]) -> torch.Tensor:
        n, size = len(subword_ids), self.window_size
        step = max(1, size - self.window_overlap)
        starts = [0]
        while starts[-1] + size < n:
            starts.append(starts[-1] + step)

        # Add [CLS] ... [SEP] to every window and pad them into one batch.
        windows = [[self.cls_id] + list(subword_ids[s:s + size]) + [self.sep_id]
                   for s in starts]
        max_len = max(len(w) for w in windows)
        pad_id = self.tokenizer.pad_token_id or 0
        input_ids = torch.full((len(windows), max_len), pad_id, dtype=torch.long)
        attention = torch.zeros((len(windows), max_len), dtype=torch.long)
        for k, w in enumerate(windows):
            input_ids[k, :len(w)] = torch.tensor(w)
            attention[k, :len(w)] = 1
        hidden = self.bert(input_ids=input_ids.to(self.device),
                           attention_mask=attention.to(self.device)).last_hidden_state

        # For each subword, take the window where it is most central.
        out = hidden.new_zeros(n, self.hidden_size)
        best = torch.full((n,), -1, dtype=torch.long)
        positions = torch.arange(n)
        for k, s in enumerate(starts):
            length = min(size, n - s)
            in_window = (positions >= s) & (positions < s + length)
            centrality = torch.minimum(positions - s, s + length - 1 - positions)
            better = in_window & (centrality > best)
            if better.any():
                idx = positions[better]
                # +1 skips the leading [CLS]
                out[idx.to(out.device)] = hidden[k, (idx - s + 1).to(out.device)]
                best[better] = centrality[better]
        return out

    def encode_label_text(self, labels: Sequence[str]) -> torch.Tensor:
        """Shared-weight label encoder: each label name -> one vector [K, d],
        the mean of its subword vectors (special tokens excluded)."""
        enc = self.tokenizer(list(labels), padding=True, return_tensors="pt")
        enc = {k: v.to(self.device) for k, v in enc.items()}
        hidden = self.bert(**enc).last_hidden_state                    # [K, T, d]
        ids = enc["input_ids"]
        # Average only the label's own subwords: skip [CLS], [SEP] and padding.
        keep = enc["attention_mask"].bool() & (ids != self.cls_id) & (ids != self.sep_id)
        mask = keep.unsqueeze(-1).float()
        return (hidden * mask).sum(1) / mask.sum(1).clamp(min=1.0)


class SpanRepresentation(nn.Module):
    """Span vector = [start token ; end token ; attention-pooled tokens] (3 x d).

    Attention pooling: a Linear(d -> 1) gives each token an importance score,
    a softmax over the span's tokens turns scores into weights, and the
    weighted average of the span's token vectors is the pooled vector.
    Works for spans of any width, so it can also represent long gold mentions.
    """

    def __init__(self, hidden_size: int):
        super().__init__()
        self.attention = nn.Linear(hidden_size, 1)

    def forward(self, word_vectors: torch.Tensor, spans: torch.Tensor) -> torch.Tensor:
        """word_vectors [n, d], spans LongTensor [S, 2] (end inclusive) -> [S, 3d]."""
        starts, ends = spans[:, 0], spans[:, 1]
        widths = ends - starts + 1
        offsets = torch.arange(int(widths.max()), device=spans.device)            # [L]
        token_idx = starts.unsqueeze(1) + offsets.unsqueeze(0)                    # [S, L]
        in_span = offsets.unsqueeze(0) < widths.unsqueeze(1)                      # [S, L]
        token_idx = token_idx.clamp(max=word_vectors.size(0) - 1)

        scores = self.attention(word_vectors).squeeze(-1)[token_idx]              # [S, L]
        scores = scores.masked_fill(~in_span, torch.finfo(scores.dtype).min)
        weights = torch.softmax(scores, dim=-1)                                   # [S, L]
        pooled = (weights.unsqueeze(-1) * word_vectors[token_idx]).sum(dim=1)     # [S, d]

        return torch.cat([word_vectors[starts], word_vectors[ends], pooled], dim=-1)


class EntityIdentifier(nn.Module):
    """Label-matching span classifier (PART 1 of the architecture)."""

    def __init__(self, model_name: str = "bert-base-cased", max_span_width: int = 8,
                 projection_hidden: int = 150, threshold: float = 0.5,
                 restrict_spans_to_sentences: bool = False, dropout: float = 0.0,
                 pos_weight: Optional[float] = None):
        super().__init__()
        self.encoder = WordEncoder(model_name)
        d = self.encoder.hidden_size
        self.span_rep = SpanRepresentation(d)
        # Span projection: 2304 -> 150 -> ReLU -> 768
        self.span_proj = nn.Sequential(
            nn.Linear(3 * d, projection_hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(projection_hidden, d),
        )
        # Label projection: 768 -> 768
        self.label_proj = nn.Linear(d, d)

        self.max_span_width = max_span_width
        self.threshold = threshold
        self.restrict_spans_to_sentences = restrict_spans_to_sentences
        self.pos_weight = pos_weight

    @property
    def device(self) -> torch.device:
        return self.encoder.device

    # ---- reusable pieces for coreference / relation extraction ----------

    def encode_label_text(self, labels: Sequence[str]) -> torch.Tensor:
        """Raw shared-BERT label vectors [K, d] (before label_proj)."""
        return self.encoder.encode_label_text(labels)

    def represent_spans(self, word_vectors: torch.Tensor, spans: Sequence) -> torch.Tensor:
        """Span vectors [S, 3d] for any (start, end) spans, e.g. gold mentions."""
        span_t = torch.as_tensor(spans, dtype=torch.long, device=word_vectors.device).view(-1, 2)
        return self.span_rep(word_vectors, span_t)

    # ---- main forward -----------------------------------------------------

    def candidate_spans(self, doc: Document) -> List:
        sent_ids = doc.sentence_ids if self.restrict_spans_to_sentences else None
        return enumerate_spans(len(doc.words), self.max_span_width, sent_ids)

    def forward(self, docs: List[Document], compute_loss: bool = True) -> Dict:
        """Score every candidate span of every document against that document's
        own entity label set.

        Returns {"docs": [per-document dict], "loss": scalar or None, "stats": ...}.
        Each per-document dict holds: doc, word_vectors [n, d], spans (list),
        span_vectors [S, 3d], logits [S, K].
        """
        # Encode every label used in this batch once, then index per document.
        all_labels = sorted({l for doc in docs for l in doc.entity_labels})
        label_pos = {l: i for i, l in enumerate(all_labels)}
        projected_labels = (self.label_proj(self.encode_label_text(all_labels))
                            if all_labels else None)

        results, loss_sum, n_elements = [], None, 0
        stats = {"gold": 0, "matched": 0, "too_long_or_filtered": 0, "unknown_label": 0}

        for doc in docs:
            word_vectors = self.encoder(doc.words)
            spans = self.candidate_spans(doc)
            out = {"doc": doc, "word_vectors": word_vectors, "spans": spans}
            if not spans or not doc.entity_labels:
                out["span_vectors"] = word_vectors.new_zeros(0, 3 * word_vectors.size(-1))
                out["logits"] = word_vectors.new_zeros(len(spans), len(doc.entity_labels))
                results.append(out)
                continue

            span_vectors = self.represent_spans(word_vectors, spans)              # [S, 2304]
            labels = projected_labels[[label_pos[l] for l in doc.entity_labels]]  # [K, 768]
            logits = self.span_proj(span_vectors) @ labels.T                      # [S, K]
            out.update(span_vectors=span_vectors, logits=logits)
            results.append(out)

            if compute_loss:
                targets, doc_stats = build_entity_targets(spans, doc.gold_mentions, doc.entity_labels)
                for k in stats:
                    stats[k] += doc_stats[k]
                targets = torch.tensor(targets, dtype=logits.dtype, device=logits.device)
                pw = (torch.tensor([self.pos_weight], dtype=logits.dtype, device=logits.device)
                      if self.pos_weight is not None else None)
                doc_loss = F.binary_cross_entropy_with_logits(
                    logits, targets, pos_weight=pw, reduction="sum")
                loss_sum = doc_loss if loss_sum is None else loss_sum + doc_loss
                n_elements += targets.numel()

        loss = loss_sum / n_elements if (compute_loss and n_elements) else None
        return {"docs": results, "loss": loss, "stats": stats}

    @torch.no_grad()
    def predict(self, docs: List[Document], threshold: Optional[float] = None) -> List[Dict]:
        """Run-time prediction: returns, per document, the kept mentions and
        their span vectors (for the coreference module)."""
        was_training = self.training
        self.eval()
        threshold = self.threshold if threshold is None else threshold
        outputs = []
        for res in self.forward(docs, compute_loss=False)["docs"]:
            probs = torch.sigmoid(res["logits"]).cpu().tolist()
            mentions = decode_mentions(res["spans"], probs, res["doc"].entity_labels,
                                       res["doc"], threshold)
            idx = torch.tensor([m["span_idx"] for m in mentions], dtype=torch.long,
                               device=res["span_vectors"].device)
            res["mentions"] = mentions
            res["mention_vectors"] = res["span_vectors"].index_select(0, idx)
            outputs.append(res)
        if was_training:
            self.train()
        return outputs

## 6. Training functions *(definitions)*

`train_model` prints sanity checks while training: dev scores before training, the
loss every 50 steps (stops if it becomes NaN), and after each epoch precision /
recall / F1, predicted vs gold mention counts and example predictions (`[ok]` =
exactly right). The best epoch is saved to `CHECKPOINT`, every epoch's scores to
`<checkpoint>_log.json`. `load_trained` loads a saved model.

In [ ]:
@torch.no_grad()
def evaluate(model, docs, threshold=None):
    """Strict entity P/R/F1, predicted/gold counts and the first document's output."""
    preds, golds, sample = [], [], None
    for batch in batches(docs, CONFIG["BATCH_SIZE"]):
        for res in model.predict(batch, threshold=threshold):
            preds.append(res["mentions"])
            golds.append(res["doc"].gold_mentions)
            if sample is None:
                sample = res
    s = entity_prf(preds, golds)
    s["n_pred"], s["n_gold"] = sum(map(len, preds)), sum(map(len, golds))
    return s, sample


def report_dev(model, dev_sets, epoch, log):
    print(f"  dev scores after epoch {epoch}:" if epoch else "  dev scores BEFORE training:")
    scores = {}
    for name, docs in dev_sets.items():
        s, sample = evaluate(model, docs)
        scores[name] = s
        print(f"    {name:9} P {s['precision']:6.2f}  R {s['recall']:6.2f}  F1 {s['f1']:6.2f}   "
              f"predicted {s['n_pred']:6} vs {s['n_gold']:6} gold")
        if epoch and s["n_gold"] and s["n_pred"] > 5 * s["n_gold"]:
            print("      ! predicting far more mentions than exist")
        if epoch and s["n_pred"] == 0:
            print("      ! predicting nothing yet")
        if epoch and sample is not None:
            doc = sample["doc"]
            gold = {(m["start"], m["end"], m["label"]) for m in doc.gold_mentions}
            print(f"    example ('{doc.doc_id}'), [ok] = exactly right:")
            for m in sample["mentions"][:8]:
                tag = "[ok]" if (m["start"], m["end"], m["label"]) in gold else "[  ]"
                text = " ".join(doc.words[m["start"]:m["end"] + 1])
                print(f"      {tag} {text!r} -> {m['label']} ({m['score']:.2f})")
    log.append({"epoch": epoch, "dev": scores})
    return sum(s["f1"] for s in scores.values()) / max(len(scores), 1)


def new_model():
    return EntityIdentifier(CONFIG["MODEL_NAME"], max_span_width=CONFIG["MAX_SPAN_WIDTH"],
                            restrict_spans_to_sentences=CONFIG["RESTRICT_SPANS_TO_SENTENCES"],
                            dropout=CONFIG["DROPOUT"], pos_weight=CONFIG["POS_WEIGHT"])


def train_model(train_docs, dev_sets):
    random.seed(CONFIG["SEED"])
    torch.manual_seed(CONFIG["SEED"])
    model = new_model().to(DEVICE)
    optimizer = torch.optim.AdamW([
        {"params": model.encoder.parameters(), "lr": CONFIG["ENCODER_LR"]},
        {"params": [p for n, p in model.named_parameters() if not n.startswith("encoder.")],
         "lr": CONFIG["HEAD_LR"]}], weight_decay=0.01)
    docs = list(train_docs)
    steps_per_epoch = math.ceil(len(docs) / CONFIG["BATCH_SIZE"])
    total = CONFIG["EPOCHS"] * steps_per_epoch
    scheduler = get_linear_schedule_with_warmup(optimizer, int(0.1 * total), total)

    log, best = [], -1.0
    report_dev(model, dev_sets, 0, log)
    for epoch in range(1, CONFIG["EPOCHS"] + 1):
        model.train()
        random.shuffle(docs)                       # mixes datasets when several are used
        start, running, n_run, unmatched = time.time(), 0.0, 0, 0
        for step, batch in enumerate(batches(docs, CONFIG["BATCH_SIZE"]), start=1):
            out = model(batch)
            if out["loss"] is None:
                continue
            if not math.isfinite(out["loss"].item()):
                raise RuntimeError(f"Loss became {out['loss'].item()}; try lower learning rates.")
            optimizer.zero_grad()
            out["loss"].backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            running += out["loss"].item()
            n_run += 1
            unmatched += out["stats"]["too_long_or_filtered"]
            if step % CONFIG["LOG_EVERY"] == 0 or step == steps_per_epoch:
                print(f"epoch {epoch} step {step}/{steps_per_epoch}: loss {running / n_run:.5f}")
                running, n_run = 0.0, 0
        print(f"epoch {epoch} took {(time.time() - start) / 60:.1f} min; {unmatched} gold "
              f"mentions had no candidate span (wider than {CONFIG['MAX_SPAN_WIDTH']} tokens)")
        mean_f1 = report_dev(model, dev_sets, epoch, log)
        if mean_f1 > best:
            best = mean_f1
            torch.save(model.state_dict(), CHECKPOINT_PATH)
            print(f"  saved best model so far (mean dev F1 {best:.2f}) to {CHECKPOINT_PATH.name}")
        LOG_PATH.write_text(json.dumps(log, indent=2))
    print(f"Done. Best mean dev F1 {best:.2f}.")
    model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=DEVICE))  # keep best epoch
    return model.eval()


def load_trained(checkpoint_path=None):
    """Load a saved model (default: CONFIG['CHECKPOINT'])."""
    path = Path(checkpoint_path or CHECKPOINT_PATH)
    if not path.exists():
        raise FileNotFoundError(f"No trained model at {path}. Run 01_entity_model.ipynb first, "
                                "or download entity_identifier.pt into this folder.")
    model = new_model()
    model.load_state_dict(torch.load(path, map_location=DEVICE))
    return model.to(DEVICE).eval()

## 7. Load the data

Training and dev documents for the datasets in the settings.

In [ ]:
train_sets, dev_sets = load_split("train"), load_split("dev")
train_docs = [doc for docs in train_sets.values() for doc in docs]
if CONFIG["QUICK"]:
    random.Random(CONFIG["SEED"]).shuffle(train_docs)
    train_docs = train_docs[:40]
    dev_sets = {k: v[:20] for k, v in dev_sets.items()}
    CONFIG["EPOCHS"] = 1
    print("QUICK MODE: 40 training docs, 20 dev docs per dataset, 1 epoch")
for name in train_sets:
    print(f"{name:9} train {len(train_sets[name]):5}   dev {len(dev_sets[name]):4}   "
          f"labels {train_sets[name][0].entity_labels}")

## 8. Train (or load the saved model)

About 20 minutes on a GPU for 10 epochs on Re-DocRED. Keep the computer awake.

In [ ]:
if CHECKPOINT_PATH.exists() and not CONFIG["RETRAIN"]:
    print(f"Loading the saved model {CHECKPOINT_PATH.name} (set RETRAIN = True to train again)")
    model = load_trained()
else:
    model = train_model(train_docs, dev_sets)
print("Model ready. Next: 03_evaluation.ipynb for test scores and the demo.")